# 1. K-Means Customer Segmentation – Online Retail


In [ ]:
!wget -q "https://archive.ics.uci.edu/static/public/352/online+retail.zip" -O online_retail.zip
!unzip -o -q online_retail.zip
!pip -q install openpyxl scikit-learn
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
df=pd.read_excel("Online Retail.xlsx")
df=df.dropna(subset=["CustomerID"])
df=df[(df.Quantity>0)&(df.UnitPrice>0)]
df["InvoiceDate"]=pd.to_datetime(df["InvoiceDate"])
df["Amount"]=df.Quantity*df.UnitPrice
snapshot=df.InvoiceDate.max()+pd.Timedelta(days=1)
rfm=df.groupby("CustomerID").agg(Recency=("InvoiceDate",lambda x:(snapshot-x.max()).days),Frequency=("InvoiceNo","nunique"),Monetary=("Amount","sum")).reset_index()
X=np.log1p(rfm[["Recency","Frequency","Monetary"]])
X=StandardScaler().fit_transform(X)


In [ ]:
inertia=[]
for k in range(2,11):
    inertia.append(KMeans(n_clusters=k,random_state=42,n_init=10).fit(X).inertia_)
plt.plot(range(2,11),inertia,marker="o")
plt.xlabel("K"); plt.ylabel("WCSS / Inertia"); plt.title("Elbow Method"); plt.show()


In [ ]:
optimal_k=4
model=KMeans(n_clusters=optimal_k,random_state=42,n_init=10)
rfm["Cluster"]=model.fit_predict(X)
display(rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean().round(2))
plt.scatter(rfm.Frequency,rfm.Monetary,c=rfm.Cluster,cmap="viridis")
plt.xlabel("Frequency"); plt.ylabel("Monetary"); plt.title("K-Means Customer Segments"); plt.show()
